# Data collection

In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
# Design family and directory
fam = '03-ext'
base_dir = r'D:\Entwicklungen\share\DataScienceProject'

In [7]:
# Import of parameters and design points
all_data = {}
fam_dir = os.path.join(base_dir, fam, 'data-' + fam)
ft_param_path = os.path.join(base_dir, 'sep24_cds_int_us-welding', 'ft_files', str(fam) + '_coll_parameters.ft')
all_data[fam] = {}

# Column names and widths
ident_cols = ['dp_no', 'mode_no', 'freq', 'rel_dev', 'mean_in_disp', 'mean_out_disp', 'mode_gain', 'mode_mac']
mode_cols = ['mode_no', 'freq', 'rel_dev', 'mean_in_disp', 'mean_out_disp', 'mode_gain', 'mode_mac']
node_cols = ['node_no', 'x_coord', 'y_coord', 'z_coord']
def_cols = ['mode' + str(x) for x in range(1, 101)]
ident_wd = [8, 12, 15, 15, 15, 15, 15, 15]
mode_wd = [8, 15, 15, 15, 15, 15, 15]
node_wd = [10, 20, 20, 20]
def_wd = [20 for x in range(100)]

# Subset of modes for each design family
fam_mode_subset = {'01': range(1, 14), '02': range(17, 31), '03': range(25, 101), '03-ext': range(25, 101)}
lst_modes = ['mode' + str(x) for x in fam_mode_subset[fam]]

# Geometrical parameters  
all_data[fam]['parameters'] = {}
params_file = os.path.join(fam_dir, 'design-points-' + fam + '.csv')
df_param = pd.read_csv(params_file, sep=';')

# Identification of the longitudinal mode
ident_file = os.path.join(fam_dir, 'mode-ident-full-' + fam + '.dat')
df_ident = pd.read_fwf(ident_file, widths=ident_wd, comment='#', names=ident_cols)

# Merge both DataFrames
all_data[fam]['parameters'] = pd.merge(df_param, df_ident, how='outer', on='dp_no')

# List of design points
lst_dp = df_param['dp_no'].to_list()

# Export DataFrame as binary file
all_data[fam]['parameters'].to_feather(ft_param_path)

In [8]:
all_data[fam]['parameters']

,dp_no,dim_x,dim_y,dim_z,nb_slots_x,slot_x_length,slot_x_distance,nb_slots_z,slot_z_length,slot_z_distance,...,cut_z_start,cut_z_end,cut_z_depth,mode_no,freq,rel_dev,mean_in_disp,mean_out_disp,mode_gain,mode_mac
0,13000,146.25,117.38,235.46,4,79.173,42.606,2,78.875,58.737,...,23.050,63.348,1.10350,77.0,19804.971,1.05774,-0.254,0.360,1.41582,0.756804
1,13001,130.53,118.96,226.27,4,82.404,54.922,2,80.518,52.995,...,32.987,66.038,4.57780,76.0,19669.124,0.44016,-0.383,0.398,1.03847,0.934646
2,13002,107.37,115.60,231.00,4,74.852,43.698,2,76.750,62.249,...,42.869,58.647,0.64307,71.0,19773.135,0.61991,-0.401,0.374,0.93259,0.922389
3,13003,116.22,123.67,257.89,4,75.324,46.253,2,79.942,65.116,...,37.865,56.437,0.31975,70.0,18595.310,0.49367,-0.374,0.368,0.98276,0.922391
4,13004,127.53,124.55,229.44,4,78.681,46.044,2,79.512,61.595,...,43.168,60.116,2.38490,69.0,18257.884,0.23684,0.346,-0.378,1.09316,0.929686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,14995,161.27,117.39,204.33,4,81.443,44.140,2,75.160,60.446,...,31.490,69.296,0.26390,78.0,19917.584,1.93442,-0.405,0.301,0.74305,0.742305
1996,14996,175.54,124.19,242.70,4,80.506,50.429,2,76.709,61.121,...,33.181,63.036,1.13220,79.0,19047.397,1.48299,0.403,-0.249,0.61718,0.798798
1997,14997,117.25,124.30,204.04,4,77.569,42.343,2,82.824,63.187,...,23.684,63.068,1.54320,68.0,18122.820,0.23480,0.413,-0.419,1.01581,0.966862
1998,14998,137.78,122.21,245.39,4,75.404,50.357,2,81.691,59.562,...,42.926,65.017,0.81244,73.0,19245.938,0.21864,0.335,-0.345,1.03126,0.967012


In [ ]:
# Import of modes, nodes and displacements for each design point
all_data[fam]['modes'] = pd.DataFrame()
all_data[fam]['nodes'] = pd.DataFrame()
all_data[fam]['defs'] = pd.DataFrame()
ft_mode_path = os.path.join(base_dir, 'sep24_cds_int_us-welding', 'ft_files', str(fam) + '_coll_modes.ft')
ft_node_path = os.path.join(base_dir, 'sep24_cds_int_us-welding', 'ft_files', str(fam) + '_coll_nodes.ft')
ft_defs_path = os.path.join(base_dir, 'sep24_cds_int_us-welding', 'ft_files', str(fam) + '_coll_defs.ft')

#for dp in [13000, 13001, 13002]:
for dp in lst_dp:
        
    print ('. Design point', str(dp), 'of', str(lst_dp[-1]))

    # Mode file
    mode_file = 'tbl-modes-' + str(dp) + '.dat'
    mode_path = os.path.join(fam_dir, 'modes', mode_file)

    if os.path.exists(mode_path):

        df_nodes = pd.read_fwf(mode_path, widths=mode_wd, comment='#', names=mode_cols)

        # Add design point as column
        n_modes = len(df_nodes)   # Number of modes
        df_nodes.insert(0, 'dp_no', pd.Series(dp*np.ones(n_modes), index=df_nodes.index))
        
        # Add subset of modes to dictionary
        all_data[fam]['modes'] = pd.concat([all_data[fam]['modes'], df_nodes[df_nodes['mode_no'].isin(fam_mode_subset[fam])]], axis=0, ignore_index=True)


    # Node and displacement files
    node_file = 'tbl-nodes-' + str(dp) + '.dat'
    node_path = os.path.join(fam_dir, 'nodes', node_file)
    def_file = 'tbl-def-y-' + str(dp) + '.dat'
    def_path = os.path.join(fam_dir, 'defs', def_file)

    if (os.path.exists(node_path) & os.path.exists(def_path)):

        # ---
        # Node file
        df_nodes = pd.read_fwf(node_path, widths=node_wd, comment='#', names=node_cols)

        # Write design point as foreign key
        node_list = df_nodes['node_no'].astype(int).to_list()
        df_nodes = pd.DataFrame({'dp_no': dp}, index=df_nodes.index).join(df_nodes)

        # Add subset of modes to dictionary
        all_data[fam]['nodes'] = pd.concat([all_data[fam]['nodes'], df_nodes], axis=0, ignore_index=True)

        # ---
        # Displacement file
        df_defs = pd.read_fwf(def_path, widths=def_wd, header=None, names=def_cols)

        # Keep the displacements of the subset of modes only
        df_defs = df_defs[lst_modes]

        # Write displacements and add "node_no", "dp_no" as foreign keys
        df_defs = pd.DataFrame({'node_no': node_list}, index=df_defs.index).join(df_defs)
        df_defs = pd.DataFrame({'dp_no': dp}, index=df_defs.index).join(df_defs)

        # Add displacements to dictionary
        all_data[fam]['defs'] = pd.concat([all_data[fam]['defs'], df_defs], axis=0, ignore_index=True)


# Specify data types
all_data[fam]['modes'] = all_data[fam]['modes'].astype({'dp_no': 'int', 'mode_no': 'int'})
all_data[fam]['modes']['rel_dev'] = pd.to_numeric(all_data[fam]['modes']['rel_dev'], errors='coerce')
all_data[fam]['nodes'] = all_data[fam]['nodes'].astype({'dp_no': 'int', 'node_no': 'int'})

# Set node coordinates in milimeters
all_data[fam]['nodes'][['x_coord', 'y_coord', 'z_coord']] = all_data[fam]['nodes'][['x_coord', 'y_coord', 'z_coord']]*1e3

# Export DataFrames as binary file
all_data[fam]['modes'].to_feather(ft_mode_path)
all_data[fam]['nodes'].to_feather(ft_node_path)
all_data[fam]['defs'].to_feather(ft_defs_path)

. Design point 13000 of 14999
. Design point 13001 of 14999
. Design point 13002 of 14999
. Design point 13003 of 14999
. Design point 13004 of 14999
. Design point 13005 of 14999
. Design point 13006 of 14999
. Design point 13007 of 14999
. Design point 13008 of 14999
. Design point 13009 of 14999
. Design point 13010 of 14999
. Design point 13011 of 14999
. Design point 13012 of 14999
. Design point 13013 of 14999
. Design point 13014 of 14999
. Design point 13015 of 14999
. Design point 13016 of 14999
. Design point 13017 of 14999
. Design point 13018 of 14999
. Design point 13019 of 14999
. Design point 13020 of 14999
. Design point 13021 of 14999
. Design point 13022 of 14999
. Design point 13023 of 14999
. Design point 13024 of 14999
. Design point 13025 of 14999
. Design point 13026 of 14999
. Design point 13027 of 14999
. Design point 13028 of 14999
. Design point 13029 of 14999
. Design point 13030 of 14999
. Design point 13031 of 14999
. Design point 13032 of 14999
. Design p

In [20]:
all_data[fam]['modes']

,dp_no,mode_no,freq,rel_dev,mean_in_disp,mean_out_disp,mode_gain,mode_mac
0,13000,25,10465.842,569.52522,0.005,0.002,0.39670,4.742000e-07
1,13000,26,10562.594,836.98470,0.003,0.001,0.30999,1.666200e-06
2,13000,27,10690.752,17.26705,-0.017,-0.050,3.01429,2.423648e-02
3,13000,28,10697.640,52.19333,0.005,0.016,3.36536,3.666563e-03
4,13000,29,10756.505,360.01716,-0.000,-0.002,12.45974,2.780962e-04
...,...,...,...,...,...,...,...,...
143787,14999,96,23755.595,1197.96751,-0.001,-0.001,0.61500,2.777900e-06
143788,14999,97,23787.878,1157.28740,-0.002,0.001,0.46326,1.735720e-05
143789,14999,98,23935.293,74.60470,-0.189,-0.016,0.08659,2.212621e-02
143790,14999,99,23998.575,1283.13848,-0.007,-0.001,0.12111,4.274050e-05


In [21]:
all_data[fam]['nodes']

,dp_no,node_no,x_coord,y_coord,z_coord
0,13000,1,-117.730000,0.0,-68.350000
1,13000,10,-117.730000,0.0,68.350000
2,13000,11,-117.730000,0.0,48.821429
3,13000,12,-117.730000,0.0,29.292857
4,13000,13,-117.730000,0.0,9.764286
...,...,...,...,...,...
827433,14999,11377,-103.647830,0.0,-22.727667
827434,14999,11389,-9.320281,0.0,45.639013
827435,14999,11393,-1.321685,0.0,-57.650627
827436,14999,11402,23.855593,0.0,52.006522


In [22]:
all_data[fam]['defs']

,dp_no,node_no,mode25,mode26,mode27,mode28,mode29,mode30,mode31,mode32,...,mode91,mode92,mode93,mode94,mode95,mode96,mode97,mode98,mode99,mode100
0,13000,1,-0.152545,0.103005,-0.431871,-0.153478,-0.001373,-0.539620,0.060917,0.260966,...,-0.195948,-0.240255,0.017833,0.043073,-0.184019,-0.524422,0.518505,-0.179127,-0.528551,0.266107
1,13000,10,-0.067810,-0.103795,-0.447014,-0.168769,-0.017965,-0.525715,-0.061969,-0.263528,...,0.186421,-0.232691,0.002575,-0.036398,-0.425681,-0.359304,-0.518142,0.173874,-0.534474,-0.255294
2,13000,11,-0.160900,-0.171439,-0.329279,-0.145528,0.022937,-0.280102,0.111420,-0.179336,...,0.240831,-0.293380,0.282648,0.099466,-0.168188,-0.564537,-0.367381,-0.026794,-0.425581,-0.545231
3,13000,12,-0.332519,-0.167854,-0.057402,-0.033475,0.045938,0.101501,0.231889,-0.116478,...,0.095839,-0.288376,0.440265,0.192407,0.306934,-0.194746,-0.054345,-0.254571,0.158219,-0.437644
4,13000,13,-0.453488,-0.067816,0.184482,0.114725,0.025156,0.365993,0.095391,-0.048896,...,-0.041690,-0.223657,0.127819,0.090189,0.458551,0.476439,0.001012,-0.087011,0.721276,-0.086181
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
827433,14999,11377,0.192485,-0.016552,0.119268,0.247655,0.076629,0.154211,0.223689,0.116655,...,-0.356012,0.050855,-0.155442,-0.011102,-0.001508,-0.254624,-0.454500,-0.172505,0.003866,-0.165610
827434,14999,11389,-0.086807,0.126193,-0.008377,0.007707,-0.059282,-0.231126,-0.016740,0.059327,...,0.419622,-0.134512,0.173201,0.019890,0.006921,-0.050741,-0.090967,-0.117269,-0.002702,0.173475
827435,14999,11393,-0.004609,-0.018034,0.000547,-0.001599,0.006163,-0.300405,-0.020531,-0.054709,...,0.406048,-0.031369,-0.012084,-0.010106,-0.005405,0.062855,0.018986,-0.213393,-0.013875,0.031907
827436,14999,11402,-0.063549,-0.247389,0.006207,-0.075778,0.186390,-0.215566,0.032333,0.034665,...,0.285050,0.250968,-0.406350,-0.012173,-0.014095,-0.099083,0.110335,-0.131153,-0.087707,0.077845
